# Flink dedup 테스트

raw 토픽에 **같은 이벤트를 두 번씩** 보내고(Active-Active pod 2개 흉내), Flink job이 clean 토픽에 한 번씩만 남기는지 확인한다.

```text
이 노트북 ──(중복 포함)──> binance.raw.spot.crypto.bookticker
                              │ DedupTestJob: (symbol, u) 처음 본 것만 통과
                              ▼
이 노트북 <── 확인 ──────── binance.clean.spot.crypto.bookticker
```

먼저 할 것은 [README.md](../README.md) 참고: 토픽 생성, port-forward, `./gradlew run`.

In [ ]:
%pip install -q kafka-python

In [ ]:
import json, time, random
from collections import Counter
from kafka import KafkaProducer, KafkaConsumer

BOOTSTRAP = "localhost:19090"
RAW_TOPIC = "binance.raw.spot.crypto.bookticker"
CLEAN_TOPIC = "binance.clean.spot.crypto.bookticker"

## 1. envelope 만들기

수집 앱의 `BinanceEventEnvelope`와 같은 모양. payload는 Spot bookTicker 원본 형태(`u`, `s`, `b`, `B`, `a`, `A`)이고, Spot bookTicker에는 `E`가 없어 `event_time`은 `None`이다.

In [ ]:
def make_envelope(symbol, u, price, collector_id):
    return {
        "source": "binance",
        "product": "spot",
        "asset_class": "crypto",
        "sector": "sov",
        "stream": "bookTicker",
        "symbol": symbol,
        "event_time": None,
        "receive_time": int(time.time() * 1000),
        "collector_id": collector_id,
        "payload": {
            "u": u, "s": symbol,
            "b": f"{price:.2f}", "B": "1.000",
            "a": f"{price + 0.01:.2f}", "A": "1.000",
        },
    }

## 2. raw에 보내기 (이벤트마다 두 번)

- 심볼 2개, 심볼마다 이벤트 `N`개
- 같은 이벤트를 `spot-sov-a`, `spot-sov-b`가 각각 보낸다. 순서는 섞는다 (두 pod 도착 시차 흉내)
- **u는 심볼마다 따로 증가**하게 일부러 같은 범위(1000부터)로 만든다. `u`만으로 dedup하면 BTC와 ETH가 서로를 지워 버린다는 걸 확인하려는 것이다

In [ ]:
SYMBOLS = {"BTCUSDT": 65000.0, "ETHUSDT": 3500.0}
N = 50
run_start_u = 1000 + int(time.time()) % 100000 * 100   # 다시 실행해도 u가 겹치지 않게

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: json.dumps(v).encode(),
    acks="all",
)

messages = []
for symbol, price in SYMBOLS.items():
    for i in range(N):
        u = run_start_u + i
        for collector in ("spot-sov-a", "spot-sov-b"):
            messages.append(make_envelope(symbol, u, price + i * 0.5, collector))
random.shuffle(messages)

for m in messages:
    producer.send(RAW_TOPIC, key=m["symbol"], value=m)
producer.flush()

expected = {(s, run_start_u + i) for s in SYMBOLS for i in range(N)}
print(f"raw로 보낸 메시지: {len(messages)}건, 기대하는 clean 건수: {len(expected)}건")

## 3. clean 확인

Flink job이 처리할 시간을 두고 clean 토픽을 읽는다. 이번 실행에서 보낸 `u` 범위만 센다.

In [ ]:
consumer = KafkaConsumer(
    CLEAN_TOPIC,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    group_id=None,                 # offset을 저장하지 않고 매번 처음부터 읽는다
    consumer_timeout_ms=15000,     # 15초 동안 새 메시지가 없으면 멈춘다
    value_deserializer=lambda v: json.loads(v),
)

received = []
for rec in consumer:
    env = rec.value
    key = (env["symbol"], env["payload"]["u"])
    if key in expected:
        received.append((key, rec.partition, env))
consumer.close()

counts = Counter(k for k, _, _ in received)
dups = {k: c for k, c in counts.items() if c > 1}
missing = expected - set(counts)

print(f"clean에서 받은 건수: {len(received)} / 기대 {len(expected)}")
print(f"중복: {len(dups)}건  {list(dups.items())[:5]}")
print(f"누락: {len(missing)}건  {sorted(missing)[:5]}")

## 4. 레코드 살펴보기

- `dedup_time`이 붙었는지
- 같은 심볼은 같은 파티션에 들어갔는지 (키 = 심볼)
- 통과한 쪽 `collector_id`는 a, b가 섞여 있다 — 먼저 도착한 쪽이 남는다

In [ ]:
for (symbol, u), partition, env in received[:3]:
    print(json.dumps(env, indent=2, ensure_ascii=False))

print("심볼별 파티션:", {s: sorted({p for (sym, _), p, _ in received if sym == s}) for s in SYMBOLS})
print("통과한 collector_id:", Counter(env["collector_id"] for _, _, env in received))
print("dedup 지연(ms) 예시:", [env["dedup_time"] - env["receive_time"] for _, _, env in received[:5]])